# 03 - Ingeniería de Datos con Pandas


En esta etapa se realiza la carga, limpieza y transformación completa 
del dataset de TechVentas S.A.S. usando Pandas. Se detectan y tratan 
los errores intencionales del dataset como valores nulos, productos en 
blanco y cantidades negativas, dejando los datos limpios y listos para 
responder preguntas de negocio.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

## 3.1 Carga del Dataset


In [ ]:
# Cargar el CSV especificando encoding y parseando fechas
tabla1 = pd.read_csv('../data/ventas_techventas.csv', 
                 encoding='latin-1', 
                 parse_dates=['fecha'])

# Corregir nombre con encoding corrupto
tabla1['vendedor'] = tabla1['vendedor'].str.replace('Ana Lï¿½ï¿', 'Ana López', regex=False)

# Explorar el dataset
print(tabla1.info())
print("─" * 50)
print(tabla1.head())

### Observaciones del dataset

- El dataset tiene 60 registros y 11 columnas
- La mayoría de columnas tienen 49 valores no nulos, lo que significa 
  que hay 11 registros con valores nulos
- La columna producto tiene 48 valores no nulos, es decir 12 nulos
- La fila 2 tiene datos mezclados en la columna order_id, lo que 
  indica un error de formato en el CSV original
- Las columnas cantidad y precio_unitario son float64 cuando 
  deberían ser int y float respectivamente

## 3.2 Limpieza de Datos

In [ ]:
# Contar nulos antes de limpiar
print("Nulos antes de limpiar:")
print(tabla1.isna().sum())
print(f"Total registros antes: {len(tabla1)}")

In [ ]:
# Eliminar filas donde producto sea nulo o esté vacío
tabla1 = tabla1[tabla1['producto'].notna() & (tabla1['producto'] != '')]

# Eliminar filas con cantidades negativas o nulas
tabla1 = tabla1[tabla1['cantidad'] > 0]

# Contar nulos después de limpiar
print("Nulos después de limpiar:")
print(tabla1.isna().sum())
print(f"Total registros después: {len(tabla1)}")
print(f"Registros eliminados: {60 - len(tabla1)}")

### Observaciones de la limpieza

- Se eliminaron 13 registros en total
- Se descartaron filas con producto nulo o vacío
- Se descartaron filas con cantidad nula o negativa
- El dataset limpio tiene 47 registros listos para el análisis
- Estrategia elegida: eliminación en vez de imputación, porque los 
  registros con nulos tenían múltiples columnas afectadas simultáneamente, 
  lo que hace inviable imputar valores confiables

## 3.3 Feature Engineering
### Creación de nuevas columnas

In [ ]:
# Crear columna ingreso_bruto
tabla1['ingreso_bruto'] = tabla1['cantidad'] * tabla1['precio_unitario']

# Crear columna ingreso_neto
tabla1['ingreso_neto'] = tabla1['ingreso_bruto'] * (1 - tabla1['descuento'])

# Crear columna mes
tabla1['mes'] = tabla1['fecha'].dt.to_period('M')

# Verificar las nuevas columnas
print(tabla1[['cantidad', 'precio_unitario', 'descuento', 'ingreso_bruto', 'ingreso_neto', 'mes']].head())

### Observaciones

- ingreso_bruto es el ingreso antes de aplicar descuentos
- ingreso_neto es el ingreso real después de descuentos
- mes extrae el período mensual de la fecha para poder 
  agrupar por mes en el siguiente paso

## 3.4 Agrupación Mensual y Cumplimiento de Metas

In [ ]:
# Agrupar por mes
resumen_mensual = tabla1.groupby('mes').agg(
    total_ingresos=('ingreso_neto', 'sum'),
    total_pedidos=('order_id', 'count')
).reset_index()

print(resumen_mensual)

In [ ]:
# Crear tabla de metas mensuales
metas = pd.DataFrame({
    'mes': pd.period_range('2024-01', '2024-06', freq='M'),
    'meta': [30000, 30000, 30000, 30000, 30000, 30000]
})

# Unir resumen mensual con metas usando merge
resumen_completo = resumen_mensual.merge(metas, on='mes')

# Calcular porcentaje de cumplimiento
resumen_completo['pct_cumplimiento'] = round(
    resumen_completo['total_ingresos'] / resumen_completo['meta'] * 100, 2
)

print(resumen_completo)

### Observaciones

- Ningún mes alcanzó el 50% de cumplimiento de la meta mensual
- Abril fue el mejor mes con 48.86% de cumplimiento
- Junio fue el peor mes con solo 27.62%
- Se observa una tendencia de caída en los últimos dos meses 
  del semestre, lo que podría ser una señal de alerta para el CEO